# Lab: Build an Agent with a LangGraph-Style State Machine

LangGraph models an application as a **graph**: nodes mutate shared state,
edges route between them, conditional edges branch on the state, and `END`
terminates. To understand exactly what the framework does for you, we build
that engine in about forty lines — then a research agent on top of it, with
tracing. The API is kept deliberately parallel to the real library, so
porting at the end is mechanical.

In [1]:
END = "__end__"

class StateGraph:
    # A minimal engine with the shape of the real thing:
    # add_node / add_edge / add_conditional_edges / invoke.
    def __init__(self):
        self.nodes, self.edges, self.conditional = {}, {}, {}
        self.entry = None

    def add_node(self, name, fn):
        self.nodes[name] = fn

    def set_entry_point(self, name):
        self.entry = name

    def add_edge(self, source, target):
        self.edges[source] = target

    def add_conditional_edges(self, source, router):
        self.conditional[source] = router

    def invoke(self, state, max_steps=12):
        state["_trace"] = []
        current = self.entry
        for step in range(1, max_steps + 1):
            state = self.nodes[current](state)
            state["_trace"].append((step, current))
            if current in self.conditional:
                current = self.conditional[current](state)
            else:
                current = self.edges.get(current, END)
            if current == END:
                return state
        state["_trace"].append(("!", "step budget exhausted"))
        return state

print("StateGraph engine defined")

StateGraph engine defined


## The research agent

Plan → search → assess → *(refine and search again | write)* → END.
The conditional edge after `assess` is where this becomes an agent rather
than a pipeline: the graph routes on what the state says about its own
progress.

In [2]:
# The knowledge base for every lab this week: support documents for Atlas
# Cycles, a fictional e-bike maker. Small enough to read, real enough to
# retrieve against.
CORPUS = {
    "battery-care": (
        "Atlas S2 battery care. Charge the battery to 80 percent for daily "
        "use and only to 100 percent before a long ride. Store between 10 "
        "and 25 degrees Celsius. A full recharge takes 4.5 hours from empty."
    ),
    "warranty": (
        "Atlas warranty policy. The frame is covered for 5 years. The "
        "battery and motor are covered for 2 years or 15,000 km, whichever "
        "comes first. Wear parts such as brake pads and tires are excluded."
    ),
    "range": (
        "Atlas S2 range guide. Expect 90 to 110 km in Eco mode, 60 to 75 km "
        "in Trail mode, and 40 to 55 km in Boost mode. Headwind, cargo "
        "weight, and cold weather reduce range by up to 30 percent."
    ),
    "error-codes": (
        "Atlas display error codes. E01 means a motor sensor fault: restart "
        "the system. E04 means battery communication lost: reseat the "
        "battery. E09 means brake cutoff engaged: check the brake levers."
    ),
    "first-service": (
        "First service. Book the complimentary first service after 300 km "
        "or 3 months. Spoke tension, brake bedding, and firmware updates "
        "are included at no charge."
    ),
}

# A deterministic embedding: character trigram counts. No model, no network,
# yet it captures enough word-shape overlap to demonstrate the geometry that
# real embedding models learn.
from collections import Counter
import math

def embed(text):
    t = " " + "".join(c.lower() if c.isalnum() else " " for c in text) + " "
    return Counter(t[i:i+3] for i in range(len(t) - 2) if t[i:i+3].strip())

def cosine(a, b):
    dot = sum(a[k] * b[k] for k in a.keys() & b.keys())
    na = math.sqrt(sum(v * v for v in a.values()))
    nb = math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

DOC_VECTORS = {name: embed(doc) for name, doc in CORPUS.items()}

def plan(state):
    state["queries"] = [state["question"]]
    state["notes"] = []
    return state

def search(state):
    q = state["queries"][-1]
    name = max(DOC_VECTORS, key=lambda n: cosine(embed(q), DOC_VECTORS[n]))
    score = cosine(embed(q), DOC_VECTORS[name])
    state["notes"].append((name, score, CORPUS[name]))
    return state

def assess(state):
    # Enough coverage when we hold two distinct sources, a strong hit, or
    # we already refined once — self-repair must be BOUNDED, or the graph
    # spends its whole step budget polishing a query it cannot improve.
    names = {n for n, _, _ in state["notes"]}
    strong = any(s >= 0.5 for _, s, _ in state["notes"])
    done_refining = state.get("refined", False)
    state["coverage"] = ("enough" if strong or len(names) >= 2
                         or done_refining else "thin")
    return state

def refine(state):
    state["queries"].append(state["question"] + " policy details terms")
    state["refined"] = True
    return state

def write(state):
    lines = [f"Research note: {state['question']}"]
    for name, score, text in state["notes"]:
        lines.append(f"- [{name}] ({score:.2f}) {text[:60]}...")
    state["report"] = "\n".join(lines)
    return state

graph = StateGraph()
for name, fn in [("plan", plan), ("search", search), ("assess", assess),
                 ("refine", refine), ("write", write)]:
    graph.add_node(name, fn)
graph.set_entry_point("plan")
graph.add_edge("plan", "search")
graph.add_edge("search", "assess")
graph.add_conditional_edges(
    "assess", lambda s: "write" if s["coverage"] == "enough" else "refine")
graph.add_edge("refine", "search")
graph.add_edge("write", END)

result = graph.invoke({"question": "what does the Atlas warranty cover"})
print(result["report"])

Research note: what does the Atlas warranty cover
- [warranty] (0.36) Atlas warranty policy. The frame is covered for 5 years. The...
- [warranty] (0.34) Atlas warranty policy. The frame is covered for 5 years. The...


## Seeing what it did

An agent you cannot trace is an agent you cannot debug or bill. The engine
recorded every step; render it the way a tracing UI would.

In [3]:
print("execution trace")
for step, node in result["_trace"]:
    print(f"  {step:>2}  {node}")
print(f"\nsearches run: {len(result['queries'])}")
for q in result["queries"]:
    print(f"  - {q}")

execution trace
   1  plan
   2  search
   3  assess
   4  refine
   5  search
   6  assess
   7  write

searches run: 2
  - what does the Atlas warranty cover
  - what does the Atlas warranty cover policy details terms


The trace shows the loop earning its keep: `assess` judged the first search
thin, `refine` expanded the query (and marked the repair *spent* — bounded
self-repair, the same discipline as yesterday's step cap), and after the
second search the graph moved on to `write`. That decision trail — which node, in what
order, why the branch went the way it went — is precisely what platforms
like LangSmith visualize for real graphs.

## Exercise — port it to the real library

```
pip install langgraph
```

1. Rebuild this graph with `langgraph.graph.StateGraph` — the node
   functions move over **unchanged**; only the wiring API differs slightly.
2. Swap `assess`'s heuristic for a model call that judges coverage, and cap
   the refine loop. Watch the trace change run to run — that variance is
   what the deterministic version deliberately hid from you.